### Initial configuration

In [1]:
import dspy
import os

API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY") 

llm = dspy.LM(model='gemini/gemini-2.5-flash', api_key=API_KEY)
        
# Configure DSPy to use this Language Model instance globally
dspy.configure(lm=llm)

### Add class-based Signatures for Screenplay Summary and quality check for it

In [2]:
import pydantic

class CriticalInfoModel(pydantic.BaseModel):
    missing_information: str
    is_important_event: bool

class OnePageSummary(dspy.Signature):
    """
    Summarize a screenplay into ONE PARAGRAPH.
    No matter how long the input is, create a concise one-paragraph summary
    that captures the essential story: main story, characters, conflict, and resolution.
    """
    screenplay = dspy.InputField(desc="Screenplay text of any length")
    summary = dspy.OutputField(
        desc="One paragraph summary (5-8 sentences) covering: main characters, conflict, key events, resolution"
    )

class SummaryQualityCheck(dspy.Signature):
    """
    Evaluate if the one-paragraph summary captures the essential information.
    """
    original_screenplay = dspy.InputField(desc="The original screenplay")
    summary = dspy.InputField(desc="The one-paragraph summary")
    
    # Quality metrics
    has_main_character = dspy.OutputField(desc="Does it mention the main character? (Yes/No)")
    has_conflict = dspy.OutputField(desc="Does it describe the main conflict? (Yes/No)")
    has_all_key_events =  dspy.OutputField(desc="Does it describe the all key events? (Yes/No)")
    has_resolution = dspy.OutputField(desc="Does it include how the story ends? (Yes/No)")
    
    information_retention_score = dspy.OutputField(
        desc="How well does it capture the essential story? Score 1-10"
    )
    missing_critical_info: list[CriticalInfoModel] = dspy.OutputField(
        desc="What important information is missing from the summary?"
    )

### Define Custom modules for Screenplay summarizer and Summary Quality validator

In [3]:
class ScreenplaySummarizer(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_summary = dspy.Predict(OnePageSummary)
    
    def forward(self, screenplay_text):
        result = self.generate_summary(screenplay=screenplay_text)
        return result


class QualityValidator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.validate = dspy.Predict(SummaryQualityCheck)
    
    def forward(self, screenplay, summary):
        result = self.validate(
            original_screenplay=screenplay,
            summary=summary
        )
        return result
    

summarizer = ScreenplaySummarizer()
summary_validator = QualityValidator()

#### Define a reusable function

In [4]:
def test_summarization(screenplay_text):
    word_count = len(screenplay_text.split())
    print(f"\nInput Size: {word_count} words")
    
    print("\nGenerating one paragraph summary...")
    result = summarizer(screenplay_text)

    print("\nONE PARAGRAPH SUMMARY:")
    print(result.summary)


    print("\nQUALITY CHECK:")
    validation = summary_validator(screenplay_text, result.summary)
    print(f"  Main Character mentioned: {validation.has_main_character}")
    print(f"  Conflict described: {validation.has_conflict}")
    print(f"  All Key Events described: {validation.has_all_key_events}")
    print(f"  Resolution included: {validation.has_resolution}")
    print(f"  Information Retention Score: {validation.information_retention_score}/10")
    
    if validation.missing_critical_info:
        print(f"\nMissing Information:")
        for info in validation.missing_critical_info:
            print(f"  - {info.missing_information}")
            print(f"  -> Is Important Event = {info.is_important_event}\n") 

In [5]:
import importlib
import data.test_data as screen_play

importlib.reload(screen_play)

test_summarization(screen_play.film_screenplays_of_8mm)


Input Size: 37312 words

Generating one paragraph summary...

ONE PARAGRAPH SUMMARY:
Private investigator Tom Welles is hired by Mrs. Christian to investigate a disturbing "snuff film" found in her deceased husband's safe, which appears to depict a real murder. Initially skeptical, Welles is horrified by the film and, against his wife Amy's wishes, embarks on a relentless quest to uncover the truth, identifying the victim as runaway Mary Anne Mathews and discovering her diary detailing abuse by her father. His investigation leads him to the seedy underworld of Los Angeles and New York, where he uncovers a network of pornographers, including Eddie Poole, Dino Velvet, and the masked killer known as "Machine," all orchestrated by Mrs. Christian's lawyer, Longdale, who commissioned the murder for a million dollars. After his young assistant Max is brutally murdered by the perpetrators, Welles, consumed by rage and a desire for justice, tracks down and violently kills Eddie Poole and then 